# dxg (cl_Dg) plotting notebook

Interactive plotting for the **optimized** `mcmcfit.py`.
Run the **Setup** cell once (it's the slow part: CAMB + reading data).
Then run any plot cell below; edit params and re-run freely. Plots display inline.

## Setup (run once)

In [ ]:
%matplotlib inline
import numpy as np
import h5py, glob, os
import matplotlib.pyplot as plt
import corner
import mcmcfit as mf

# ---- EDIT these ----
DATA_DIR = "../dxg_measurement"          # folder with the 5 real .h5 files
FRB_CATALOG = None                       # path to cat_II_..._inclusion.h5, or None for stand-in

# ---- load files (redshift order, low z first) ----
FILES = sorted(glob.glob(os.path.join(DATA_DIR, "*.h5")))
assert len(FILES) == 5, f"expected 5, found {len(FILES)}: {FILES}"
print("Using:", [os.path.basename(f) for f in FILES])

if FRB_CATALOG:
    with h5py.File(FRB_CATALOG, "r") as fh:
        dm_select = fh["dm_od"][:] + fh["dm_ave"][()]
else:
    dm_select = np.random.default_rng(0).normal(500, 200, 500)
    print("WARNING: stand-in dm_select (no real FRB catalog).")

g = mf.cl_Dg_fit(dm_select)
g.read_data(FILES, FILES)   # (fit_files, plot_files); same list if no separate plot files
print("Loaded. nz =", g.nz, " lmax =", g.lmax, " zs =", g.zs)

## Current parameters
Edit and re-run any plot after changing these. Order: `DM_H_bar, cut_scale, l_cut, bf0, ALPHA, Z_STAR`

In [ ]:
DM_H_bar, cut_scale, l_cut, bf0, ALPHA, Z_STAR = 60.0, -1.0, 4000.0, 1.5, -1.5, 1.0
params = (DM_H_bar, cut_scale, l_cut, bf0, ALPHA, Z_STAR)

## pf(chi) — FRB radial distribution

In [ ]:
pf_z, n = g.get_p_z_norm(ALPHA, Z_STAR)
pf_chi = 4*np.pi*g.chi_grid**2*n/g.Nf
plt.figure(figsize=(6,4))
plt.plot(g.chi_grid, pf_chi)
plt.xlabel(r'$\chi$ [Mpc]'); plt.ylabel(r'$p_f(\chi)$ [Mpc$^{-1}$]')
plt.title(f'FRB radial distribution (ALPHA={ALPHA}, Z_STAR={Z_STAR})')
plt.show()

## Model vs data — all 5 z bins (built-in plot_models)

In [ ]:
g.plot_models(*params, mode='full', convolved=False, save=False)
plt.show()

## Model Cl with components (total / background / same-halo), one z bin

In [ ]:
z_ind = 2   # 0..4 ; bins 0,1 are sub-bins (weighted), 2,3,4 single-redshift
l = np.linspace(1, g.lmax, g.lmax)
cl_Dg, term1, term2 = g.model_cl_Dg_plot(l, z_ind, *params)
plt.figure(figsize=(6,4))
plt.errorbar(g.l_fit[z_ind], g.l_fit[z_ind]*g.cl_fit[z_ind],
             yerr=g.l_fit[z_ind]*g.err_fit[z_ind], fmt='.', capsize=2, label='data')
plt.semilogx(l, l*cl_Dg, label='total')
plt.semilogx(l, l*term1, '--', label='background')
plt.semilogx(l, l*term2, '-.', label='same halo')
plt.axhline(0, color='k', lw=0.5, ls=':')
plt.xlabel(r'$\ell$'); plt.ylabel(r'$\ell C_\ell$')
plt.title(f'z bin {z_ind}, z = {g.zs[z_ind]:.3f}')
plt.legend(); plt.show()

## Corner plot
Either read an existing `mcmc_out.h5`, **or** run a short fit. Editing `MCMC_OUT`.

In [ ]:
MCMC_OUT = None   # set to "path/to/mcmc_out.h5" to read existing samples
LABELS = ["<DM_H>", "s_cut", "l_cut", "bf", "ALPHA", "Z_STAR"]

if MCMC_OUT:
    with h5py.File(MCMC_OUT, "r") as f:
        flat_samples = f["flat_samples"][:]
        if "parameter_names" in f:
            LABELS = [s.decode() if isinstance(s, bytes) else s for s in f["parameter_names"][:]]
else:
    g.mcmc(nsteps=2000, burn_in=500, thin=10, save=False)  # raise nsteps for a real posterior
    flat_samples = g.flat_samples

fig = corner.corner(flat_samples, labels=LABELS)
plt.show()